# Notebook 04 — Chatbot Completo Integrado

**Curso:** Minería de Textos | **Proyecto 3** | CUC

---

## Orden de ejecución
```
Notebook 02 → Fine-Tuning + corpus etiquetado
Notebook 03 → RAG Pipeline + índice FAISS
Notebook 04 → Chatbot integrado (este notebook)
```

## Mejoras implementadas
- Filtro de emoción solo activo si score >= 70% (evita filtros incorrectos)
- Generador Flan-T5 en dos pasos: genera en inglés → traduce al español
- Memoria conversacional de los últimos 5 turnos


In [1]:
import sys, json
sys.path.append('../app')
sys.path.append('../src')

from pathlib import Path

from src.mongo_utils      import mongo_utils
from src.finetuning_utils import finetuning_utils
from src.rag_utils        import rag_utils
from src.chatbot_engine   import chatbot_engine

print('Librerías cargadas.')

INFO:app.config:✅ Configuración cargada. Proveedor de LLM: local
C:\Users\pmari\OneDrive\Para Revisar\Documentos\GitHub\chatbot-musical-inteligente\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:faiss.loader:Loading faiss with AVX2 support.
INFO:faiss.loader:Successfully loaded faiss with AVX2 support.


Librerías cargadas.


## 1. Inicializar sistema completo

In [2]:
db_conexion = mongo_utils()
finetuning  = finetuning_utils()
rag         = rag_utils()

if db_conexion.verificar_conexion():
    canciones_raw     = db_conexion.cargar_canciones()
    corpus_etiquetado = finetuning.etiquetar_corpus_con_modelo(canciones_raw)
    rag.inicializar(corpus_etiquetado)
    bot = chatbot_engine()
    print('\nSistema listo: Fine-Tuning + RAG + Chatbot integrados.')
else:
    print('ERROR: No se pudo conectar a MongoDB Atlas.')

DEBUG:mongo_utils:mongo_utils instanciado.
DEBUG:EmotionClassifier:EmotionClassifier instanciado.
DEBUG:rag_utils:rag_utils instanciado.


2026-04-23 09:14:19 | INFO     | mongo_utils | Conectando a MongoDB Atlas | DB: analisisMusical | Col: analisisMusical


INFO:mongo_utils:Conectando a MongoDB Atlas | DB: analisisMusical | Col: analisisMusical
DEBUG:mongo_utils:Cliente MongoClient creado.


2026-04-23 09:14:20 | INFO     | mongo_utils | Conexión verificada correctamente.


INFO:mongo_utils:Conexión verificada correctamente.


...

2026-04-23 09:14:22 | INFO     | chatbot_engine | Cargando google/flan-t5-large en CUDA...


INFO:chatbot_engine:Cargando google/flan-t5-large en CUDA...
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/flan-t5-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-large/0613663d0d48ea86ba8cb3d7a44f0f65dc596a2a/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/google/flan-t5-large/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-large/0613663d0d48ea86ba8cb3d7a44f0f65dc596a2a/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-large/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"

2026-04-23 09:14:32 | INFO     | chatbot_engine | Flan-T5 cargado correctamente.


INFO:chatbot_engine:Flan-T5 cargado correctamente.



Sistema listo: Fine-Tuning + RAG + Chatbot integrados.


## 2. Prueba interactiva con emoción y umbral de confianza

El filtro de emoción solo se activa si el clasificador tiene >= 70% de confianza.
Esto evita filtros incorrectos como clasificar 'alegría' como 'rabia' con baja confianza.


In [3]:
def chat(pregunta):
    resultado = bot.responder(pregunta)
    emocion   = resultado.get('emocion')
    print(f'Usuario: {pregunta}')
    if emocion:
        activo = emocion['score'] >= 0.70
        print(f'[Emoción: {emocion["emocion"]} ({emocion["score"]:.0%}) '
              f'| Filtro: {"activo" if activo else "inactivo (score bajo)"}]')
    print(f'Bot: {resultado["respuesta"]}')
    fuentes = [c['titulo'] for c in resultado['chunks'][:3]]
    print(f'[Fuentes: {", ".join(fuentes)}]')
    print()
    return resultado

# Conversación de prueba con memoria
chat('Estoy triste, que cancion me recomiendas?')
chat('Dame otra similar.')
chat('De que artista es la primera que mencionaste?')

2026-04-23 09:15:10 | INFO     | chatbot_engine | Pregunta recibida: 'Estoy triste, que cancion me recomiendas?'


INFO:chatbot_engine:Pregunta recibida: 'Estoy triste, que cancion me recomiendas?'
DEBUG:chatbot_engine:Intención: recomendacion
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 4037.09it/s]


2026-04-23 09:15:13 | INFO     | EmotionClassifier | Clasificador cargado para inferencia.


INFO:EmotionClassifier:Clasificador cargado para inferencia.
DEBUG:EmotionClassifier:Emoción predicha: {'emocion': 'tristeza', 'score': 0.2667}
DEBUG:rag_utils:Buscando | pregunta='Estoy triste, que cancion me recomiendas?' | top_k=5 | emocion=None


2026-04-23 09:15:13 | INFO     | rag_utils | Cargando modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


...

INFO:chatbot_engine:Pregunta recibida: 'De que artista es la primera que mencionaste?'
DEBUG:chatbot_engine:Intención: seguimiento
DEBUG:EmotionClassifier:Emoción predicha: {'emocion': 'rabia', 'score': 0.2571}
DEBUG:chatbot_engine:Búsqueda enriquecida: De que artista es la primera que mencionaste? Green Light (HBA Remix) Beyoncé po
DEBUG:rag_utils:Buscando | pregunta='De que artista es la primera que mencionaste? Gree' | top_k=5 | emocion=None
Batches: 100%|██████████| 1/1 [00:00<00:00, 45.54it/s]
DEBUG:rag_utils:Resultados encontrados: 5
DEBUG:chatbot_engine:Chunks recuperados: 5
DEBUG:chatbot_engine:Flan-T5 [seguimiento]: Taylor Swift
DEBUG:chatbot_engine:Respuesta inválida detectada, usando fallback.


2026-04-23 09:15:26 | INFO     | chatbot_engine | Respuesta generada (149 chars)


INFO:chatbot_engine:Respuesta generada (149 chars)


Usuario: De que artista es la primera que mencionaste?
[Emoción: rabia (26%) | Filtro: inactivo (score bajo)]
Bot: Te recomiendo "The Last Time" de Taylor Swift, una canción de pop con emoción de nostalgia. También puedes escuchar "Green Light (Remix)" de Beyoncé.
[Fuentes: The Last Time, Green Light (Remix), The Last Time]



{'respuesta': 'Te recomiendo "The Last Time" de Taylor Swift, una canción de pop con emoción de nostalgia. También puedes escuchar "Green Light (Remix)" de Beyoncé.',
 'chunks': [{'texto': "pre taylor swift gary lightbody and right before your eyes i'm aching run fast nowhere to hide just you and me taylor swift gary lightbody this is the last time i'm asking you this put my name at the top of your list this is the last time i'm asking",
   'titulo': 'The Last Time',
   'artista': 'Taylor Swift',
   'genero': 'pop',
   'anio': 2012,
   'idioma': 'en',
   'emocion': 'nostalgia',
   'emocion_score': 0.9631,
   'score': 0.22960466146469116},
  {'texto': "along go go go oh oh you got the green light so you can go go go go oh oh go go red light green light beyoncé i gave all i could give my love my heart now we're facing the end from what you did from the start my",
   'titulo': 'Green Light (Remix)',
   'artista': 'Beyoncé',
   'genero': 'pop',
   'anio': 2007,
   'idioma': 'en',
   'emoci

## 3. Generar 10 conversaciones de prueba

In [4]:
bot.limpiar_historial()

conversaciones = [
    # Factuales
    'Que canciones hablan de desamor?',
    'Que artistas de pop hay en el corpus?',
    # Emocionales
    'Estoy muy feliz hoy, que cancion me recomiendas?',
    'Necesito algo para llorar un rato.',
    # Comparativas
    'Que diferencia hay entre una balada triste y una alegre?',
    # Seguimiento (prueba de memoria)
    'Dame otra cancion parecida a la ultima que mencionaste.',
    'Y esa de quien es?',
    # Por género
    'Dame una cancion de pop con emocion de amor.',
    # Fuera de dominio (el bot debe decir que no sabe)
    'Cuanto cuesta un vuelo a Madrid?',
    'Quien gano el mundial de futbol en 2022?',
]

registro = []
for i, pregunta in enumerate(conversaciones, 1):
    print(f'--- Conversación {i} ---')
    resultado = chat(pregunta)
    registro.append({
        'id':       i,
        'pregunta': pregunta,
        'respuesta': resultado['respuesta'],
        'emocion_detectada': resultado.get('emocion'),
        'fuentes': [
            {'titulo': c['titulo'], 'artista': c['artista'],
             'emocion': c['emocion'], 'score': c['score']}
            for c in resultado['chunks']
        ],
    })

# Guardar en metricas.json
Path('../resultados').mkdir(exist_ok=True)
metricas_path = Path('../resultados/metricas.json')
metricas_existentes = {}
if metricas_path.exists():
    with open(metricas_path, 'r', encoding='utf-8') as f:
        metricas_existentes = json.load(f)

metricas_existentes['conversaciones_prueba'] = registro
with open(metricas_path, 'w', encoding='utf-8') as f:
    json.dump(metricas_existentes, f, ensure_ascii=False, indent=2)
print('\n10 conversaciones guardadas en resultados/metricas.json')

2026-04-23 09:15:36 | INFO     | chatbot_engine | Historial conversacional limpiado.


INFO:chatbot_engine:Historial conversacional limpiado.


--- Conversación 1 ---
2026-04-23 09:15:36 | INFO     | chatbot_engine | Pregunta recibida: 'Que canciones hablan de desamor?'


INFO:chatbot_engine:Pregunta recibida: 'Que canciones hablan de desamor?'
DEBUG:chatbot_engine:Intención: recomendacion
DEBUG:EmotionClassifier:Emoción predicha: {'emocion': 'tristeza', 'score': 0.3891}
DEBUG:rag_utils:Buscando | pregunta='Que canciones hablan de desamor?' | top_k=5 | emocion=None
Batches: 100%|██████████| 1/1 [00:00<00:00, 20.63it/s]
DEBUG:rag_utils:Resultados encontrados: 5
DEBUG:chatbot_engine:Chunks recuperados: 5
DEBUG:chatbot_engine:Flan-T5 [recomendacion]: Te recomiendo
DEBUG:chatbot_engine:Respuesta inválida detectada, usando fallback.


2026-04-23 09:15:38 | INFO     | chatbot_engine | Respuesta generada (143 chars)


...

Usuario: Cuanto cuesta un vuelo a Madrid?
[Emoción: tristeza (28%) | Filtro: inactivo (score bajo)]
Bot: This is about music.
[Fuentes: ​​rockstar (Remix), Airplane pt.2, Te Dejo Madrid]

--- Conversación 10 ---
2026-04-23 09:15:52 | INFO     | chatbot_engine | Pregunta recibida: 'Quien gano el mundial de futbol en 2022?'


INFO:chatbot_engine:Pregunta recibida: 'Quien gano el mundial de futbol en 2022?'
DEBUG:chatbot_engine:Intención: fuera_dominio
DEBUG:EmotionClassifier:Emoción predicha: {'emocion': 'amor', 'score': 0.2294}
DEBUG:rag_utils:Buscando | pregunta='Quien gano el mundial de futbol en 2022?' | top_k=5 | emocion=None
Batches: 100%|██████████| 1/1 [00:00<00:00, 50.79it/s]
DEBUG:rag_utils:Resultados encontrados: 5
DEBUG:chatbot_engine:Chunks recuperados: 5
DEBUG:chatbot_engine:Flan-T5 [fuera_dominio]: 'Lo siento, solo puedo ayudarte con msica. Te recomenda una canción?'


2026-04-23 09:15:58 | INFO     | chatbot_engine | Respuesta generada (69 chars)


INFO:chatbot_engine:Respuesta generada (69 chars)


Usuario: Quien gano el mundial de futbol en 2022?
[Emoción: amor (23%) | Filtro: inactivo (score bajo)]
Bot: 'Lo siento, solo puedo ayudarte con msica. Te recomenda una canción?'
[Fuentes: The 1989 World Tour - Special Guests, Up All Night, Up All Night]


10 conversaciones guardadas en resultados/metricas.json


## 4. Prueba de memoria conversacional

In [5]:
bot.limpiar_historial()
print('=== PRUEBA DE MEMORIA CONVERSACIONAL ===')
chat('Dame una cancion triste de pop.')
chat('Puedes decirme mas sobre esa cancion?')
chat('Del mismo artista hay algo mas alegre?')
print(f'Turnos en memoria: {len(bot.historial)}')

2026-04-23 09:16:18 | INFO     | chatbot_engine | Historial conversacional limpiado.


INFO:chatbot_engine:Historial conversacional limpiado.


=== PRUEBA DE MEMORIA CONVERSACIONAL ===
2026-04-23 09:16:18 | INFO     | chatbot_engine | Pregunta recibida: 'Dame una cancion triste de pop.'


INFO:chatbot_engine:Pregunta recibida: 'Dame una cancion triste de pop.'
DEBUG:chatbot_engine:Intención: recomendacion
DEBUG:EmotionClassifier:Emoción predicha: {'emocion': 'amor', 'score': 0.3582}
DEBUG:rag_utils:Buscando | pregunta='Dame una cancion triste de pop.' | top_k=5 | emocion=None
Batches: 100%|██████████| 1/1 [00:00<00:00, 21.38it/s]
DEBUG:rag_utils:Resultados encontrados: 5
DEBUG:chatbot_engine:Chunks recuperados: 5
DEBUG:chatbot_engine:Flan-T5 [recomendacion]: Te recomiendo
DEBUG:chatbot_engine:Respuesta inválida detectada, usando fallback.


2026-04-23 09:16:20 | INFO     | chatbot_engine | Respuesta generada (139 chars)


...

Usuario: Puedes decirme mas sobre esa cancion?
[Emoción: amor (23%) | Filtro: inactivo (score bajo)]
Bot: Te recomiendo "1997 Freestyle Live at Wetlands, NYC" de Eminem, una canción de hip hop con emoción de rabia. También puedes escuchar "Family Matters" de Drake.
[Fuentes: 1997 Freestyle Live at Wetlands, NYC, Family Matters, LA CANCIÓN]

2026-04-23 09:16:22 | INFO     | chatbot_engine | Pregunta recibida: 'Del mismo artista hay algo mas alegre?'


INFO:chatbot_engine:Pregunta recibida: 'Del mismo artista hay algo mas alegre?'
DEBUG:chatbot_engine:Intención: seguimiento
DEBUG:EmotionClassifier:Emoción predicha: {'emocion': 'rabia', 'score': 0.239}
DEBUG:chatbot_engine:Búsqueda enriquecida: Del mismo artista hay algo mas alegre? 1997 Freestyle Live at Wetlands, NYC Emin
DEBUG:rag_utils:Buscando | pregunta='Del mismo artista hay algo mas alegre? 1997 Freest' | top_k=5 | emocion=None
Batches: 100%|██████████| 1/1 [00:00<00:00, 67.11it/s]
DEBUG:rag_utils:Resultados encontrados: 5
DEBUG:chatbot_engine:Chunks recuperados: 4
DEBUG:chatbot_engine:Flan-T5 [seguimiento]: Te recomiendo
DEBUG:chatbot_engine:Respuesta inválida detectada, usando fallback.


2026-04-23 09:16:24 | INFO     | chatbot_engine | Respuesta generada (168 chars)


INFO:chatbot_engine:Respuesta generada (168 chars)


Usuario: Del mismo artista hay algo mas alegre?
[Emoción: rabia (24%) | Filtro: inactivo (score bajo)]
Bot: Te recomiendo "Baruch College Session" de Eminem, una canción de hip hop con emoción de nostalgia. También puedes escuchar "Rolling Stone Q/A Exclusive 2013" de Eminem.
[Fuentes: Baruch College Session, Rolling Stone Q/A Exclusive 2013, 힙합성애자 (Hip Hop Lover/Phile)]

Turnos en memoria: 3


## 5. Análisis del umbral de confianza

Verificar cuántas preguntas activan el filtro de emoción (score >= 70%).


In [7]:
preguntas_test = [
    'Estoy muy triste hoy, me rompieron el corazón y no puedo dejar de llorar',
    'Quiero bailar y celebrar, estoy muy feliz y con mucha energía',
    'Me siento nostálgico recordando el pasado y los viejos tiempos',
    'Dame algo energético y alegre para una fiesta con amigos',
    'Siento mucha rabia y odio, me traicionaron y quiero desahogarme',
    'Busco una canción romántica de amor para dedicarle a alguien especial',
]

print('ANÁLISIS DE UMBRAL DE CONFIANZA (>= 70% activa filtro)')
print('=' * 60)
for p in preguntas_test:
    res = finetuning.predecir_emocion(p)
    if res:
        activo = 'ACTIVO' if res['score'] >= 0.70 else 'inactivo'
        print(f'{activo} | {res["emocion"]:12s} {res["score"]:.0%} | "{p[:50]}..."')

DEBUG:EmotionClassifier:Emoción predicha: {'emocion': 'tristeza', 'score': 0.9437}


ANÁLISIS DE UMBRAL DE CONFIANZA (>= 70% activa filtro)
ACTIVO | tristeza     94% | "Estoy muy triste hoy, me rompieron el corazón y no..."


DEBUG:EmotionClassifier:Emoción predicha: {'emocion': 'alegria', 'score': 0.8933}
DEBUG:EmotionClassifier:Emoción predicha: {'emocion': 'nostalgia', 'score': 0.558}
DEBUG:EmotionClassifier:Emoción predicha: {'emocion': 'alegria', 'score': 0.769}
DEBUG:EmotionClassifier:Emoción predicha: {'emocion': 'rabia', 'score': 0.51}
DEBUG:EmotionClassifier:Emoción predicha: {'emocion': 'amor', 'score': 0.7729}


ACTIVO | alegria      89% | "Quiero bailar y celebrar, estoy muy feliz y con mu..."
inactivo | nostalgia    56% | "Me siento nostálgico recordando el pasado y los vi..."
ACTIVO | alegria      77% | "Dame algo energético y alegre para una fiesta con ..."
inactivo | rabia        51% | "Siento mucha rabia y odio, me traicionaron y quier..."
ACTIVO | amor         77% | "Busco una canción romántica de amor para dedicarle..."
